# VILAGENT Text LLM via vLLM + pyngrok

Run this notebook on a Colab GPU runtime. It starts an OpenAI-compatible vLLM server and exposes it through pyngrok for `VILAGENT_TEXT_PYNGROK_BASE_URL`.

In [ ]:
!pip -q install -U pyngrok vllm


In [ ]:
import os
from getpass import getpass

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN") or getpass("NGROK_AUTHTOKEN: ")
VLLM_API_KEY = os.getenv("VLLM_API_KEY") or getpass("VLLM_API_KEY for VILAGENT: ")
MODEL_NAME = os.getenv("VILAGENT_TEXT_MODEL_NAME", "Qwen/Qwen3-32B")
PORT = int(os.getenv("VILAGENT_TEXT_PORT", "8000"))

os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["VLLM_API_KEY"] = VLLM_API_KEY
print({"model": MODEL_NAME, "port": PORT})


In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--model", MODEL_NAME,
    "--api-key", VLLM_API_KEY,
]
server = subprocess.Popen(cmd)
time.sleep(15)
print("vLLM server started with pid", server.pid)


In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url.rstrip("/")
print("Public base:", public_url)
print("\nCopy into .env:")
print("VILAGENT_TEXT_MODEL_PROVIDER=pyngrok")
print("VILAGENT_TEXT_MODEL_CONFIG_NAME=vilagent-text-pyngrok")
print(f"VILAGENT_TEXT_MODEL_NAME={MODEL_NAME}")
print(f"VILAGENT_TEXT_API_KEY={VLLM_API_KEY}")
print(f"VILAGENT_TEXT_PYNGROK_BASE_URL={public_url}/v1")


In [ ]:
import requests

headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
res = requests.get(f"{public_url}/v1/models", headers=headers, timeout=30)
print(res.status_code)
print(res.text[:1000])
